## We define the known sample moments

In [1]:
# Mean
μ = var('μ')
# Standard deviation
σ = var('σ')
# Skewness
γ = var('γ')
# Excess kurtosis
κ = var('κ')
μprime = function("μ'")
def sample_moment(m, size):
    if m == 0:
        return 1
    elif m == 1:
        return μ
    elif m == 2:
        return μ^2 + σ^2/size
    elif m == 3:
        return μ^3 + 3*μ*σ^2/size + γ*σ^3/size^2
    elif m == 4:
        return μ^4 + 6*μ^2*σ^2/size + 4*γ*μ*σ^3/size^2 + (κ + 3*size)*σ^4/size^3
    else:
        return μprime(m, size)

In [2]:
sample_moment(4, var('N'))

μ^4 + 6*μ^2*σ^2/N + 4*γ*μ*σ^3/N^2 + (3*N + κ)*σ^4/N^3

## We define variables

In [3]:
def get_regular_variables(num, prefix='n_'):
    var_list = ','.join([(prefix + str(n)) for n in range(1, num + 1)])
    gens = var(var_list)
    return_vars = dict()
    for n in range(0, num):
        return_vars[n + 1] = gens[n]
    return return_vars

In [4]:
base_ring = (var('stupid')).parent()
def get_variables(num, prefix='n_', ring=base_ring):
    var_list = ','.join([(prefix + str(n)) for n in range(1, num + 1)])
    gens = ring[var_list].gens()
    return_vars = dict()
    for n in range(0, num):
        return_vars[n + 1] = gens[n]
    return return_vars

In [9]:
def get_Xs(num):
    return get_variables(num, prefix='X_')

In [15]:
def get_Xbar(X, N):
    return sum([N[g]*X[g] for g in X.keys()]) / sum(N.values())

## We define expectation

In [11]:
def expectation(expression):
    "expression should be an element of R"
    result = 0
    for coefficient, monomial in expression:
        new_term = coefficient
        exponents = monomial.exponents()[0]
        for index, exponent in enumerate(exponents):
            new_term *= sample_moment(exponent, N[index + 1])
        result += new_term
    return result

In [20]:
Xbar.substitute?

In [24]:
G = 3
NN = var('N')
N = get_regular_variables(G, 'N_')
X = get_Xs(G)
Xbar = get_Xbar(X, N)
substitution_dict = {
    N[1]: (NN - sum(N.values()) + N[1]).full_simplify()
}
for expr in [Xbar^k for k in range(1, 5)]:
    e1 = expectation(expr).full_simplify().substitute(substitution_dict).full_simplify()
    show(e1)

μ

(N*μ^2 + σ^2)/N

(N^2*μ^3 + 3*N*μ*σ^2 + γ*σ^3)/N^2

(N^3*μ^4 + 6*N^2*μ^2*σ^2 + 4*N*γ*μ*σ^3 + (3*N + κ)*σ^4)/N^3

In [26]:
def get_V(u):
    G = len(u)
    NN = var('N')
    N = get_regular_variables(G, 'N_')
    allN = sum(N.values())
    X = get_Xs(G)
    Xbar = get_Xbar(X, N)
    Vu = NN*sum([
        u[g]*N[g]/(allN-N[g])*(X[g] - Xbar)^2
        for g in range(1, G+1)
    ])
    return (NN, N, X, Xbar, Vu)
    
G = 3
u = get_regular_variables(G, 'u_')
NN, N, X, Xbar, V = get_V(u)
show(V)

((N_1*u_1*(N_1/(N_1 + N_2 + N_3) - 1)^2/(N_2 + N_3) + N_1^2*N_2*u_2/((N_1 + N_2 + N_3)^2*(N_1 + N_3)) + N_1^2*N_3*u_3/((N_1 + N_2 + N_3)^2*(N_1 + N_2)))*N)*X_1^2 + (2*(N_1*N_2*u_1*(N_1/(N_1 + N_2 + N_3) - 1)/((N_1 + N_2 + N_3)*(N_2 + N_3)) + N_1*N_2*u_2*(N_2/(N_1 + N_2 + N_3) - 1)/((N_1 + N_2 + N_3)*(N_1 + N_3)) + N_1*N_2*N_3*u_3/((N_1 + N_2 + N_3)^2*(N_1 + N_2)))*N)*X_1*X_2 + ((N_2*u_2*(N_2/(N_1 + N_2 + N_3) - 1)^2/(N_1 + N_3) + N_1*N_2^2*u_1/((N_1 + N_2 + N_3)^2*(N_2 + N_3)) + N_2^2*N_3*u_3/((N_1 + N_2 + N_3)^2*(N_1 + N_2)))*N)*X_2^2 + (2*(N_1*N_3*u_1*(N_1/(N_1 + N_2 + N_3) - 1)/((N_1 + N_2 + N_3)*(N_2 + N_3)) + N_1*N_3*u_3*(N_3/(N_1 + N_2 + N_3) - 1)/((N_1 + N_2 + N_3)*(N_1 + N_2)) + N_1*N_2*N_3*u_2/((N_1 + N_2 + N_3)^2*(N_1 + N_3)))*N)*X_1*X_3 + (2*(N_2*N_3*u_2*(N_2/(N_1 + N_2 + N_3) - 1)/((N_1 + N_2 + N_3)*(N_1 + N_3)) + N_2*N_3*u_3*(N_3/(N_1 + N_2 + N_3) - 1)/((N_1 + N_2 + N_3)*(N_1 + N_2)) + N_1*N_2*N_3*u_1/((N_1 + N_2 + N_3)^2*(N_2 + N_3)))*N)*X_2*X_3 + ((N_3*u_3*(N_3/(N_1 + N_2 + N_3) - 1)^2/(N_1 + N_2) + N_1*N_3^2*u_1/((N_1 + N_2 + N_3)^2*(N_2 + N_3)) + N_2*N_3^2*u_2/((N_1 + N_2 + N_3)^2*(N_1 + N_3)))*N)*X_3^2

In [32]:
for G in range(2, 7):
    u = get_regular_variables(G, 'u_')
    NN, N, X, Xbar, V = get_V(u)
    substitution_dict = {
        N[1]: (NN - sum(N.values()) + N[1]).full_simplify(),
        sum(u.values()): 1
    }
    exp = expectation(V).full_simplify().substitute(substitution_dict).full_simplify().substitute(substitution_dict).full_simplify()
    if exp != σ^2:
        print(G, exp)

In [33]:
def get_NN_N_X_Xbar_u_V(G):
    u = get_regular_variables(G, 'u_')
    NN, N, X, Xbar, V = get_V(u)
    return (NN, N, X, Xbar, u, V)

## Hypothesis

### Build hypothesis

We pull out a coefficient of u_1^n

In [ ]:
varV__u_1_coeff = varV.coefficient(u_1, 2)
varV__u_1_coeff = varV__u_1_coeff.factor().collect(θ_2).simplify()
print(varV__u_1_coeff)
show(varV__u_1_coeff)

Pull $θ_2^2$ coeff of $u_1^2$ coeff

In [ ]:
varV__u_1__θ_2 = varV__u_1_coeff.coefficient(θ_2, 2)
print(varV__u_1__θ_2)
show(varV__u_1__θ_2)

In [ ]:
# varV__u_1__θ_2_hyp = (2*n[1]-3)*(1/n[1] - 1/N) + 2 * n[1] / N + 3 / (n[1] * (N - n[1]))
varV__u_1__θ_2_hyp = (2*n[1]-3)*(1/n[1] - 1/N) + 2*n[1]/N + 3*(1/N - n[1]/(N*(N-n[1])))
zero = (varV__u_1__θ_2 - varV__u_1__θ_2_hyp).full_simplify()
zero

In [ ]:
varV__u_1__θ_4 = varV__u_1_coeff.coefficient(θ_4, 1)
print(varV__u_1__θ_4)
show(varV__u_1__θ_4)

In [ ]:
varV__u_1__θ_4_hyp = N/(n[1]*(N-n[1])) - 3/N
zero = (varV__u_1__θ_4 - varV__u_1__θ_4_hyp).full_simplify()
zero

In [ ]:
varV__u_1_coeff_hyp = (N/(n[1]*(N-n[1])) - 3/N) * θ_4 + ((2*n[1]-3)*(1/n[1] - 1/N) + 2*n[1]/N + 3*(1/N - n[1]/(N*(N-n[1])))) * θ_2^2
zero = (varV__u_1_coeff - varV__u_1_coeff).full_simplify()
zero

Now we try an off-diagonal term, a coefficient of $u_1 u_2$.

In [ ]:
varV__u_12_coeff = varV.coefficient(u_1*u_2, 1)
varV__u_12_coeff = varV__u_12_coeff.factor().collect(θ_2).simplify()
print(varV__u_12_coeff)
show(varV__u_12_coeff)

In [ ]:
# varV__u_12_coeff_hyp =  ((2 - 3/N)*(1/(N - n[1]) + 1/(N - n[2])) - N/((N-n[1])*(N-n[2]))) * θ_4 + (N*(2*N + 3)/((N - n[1])*(N - n[2])) + 2 + 9/N - 2*(N+3)*(1/(N - n[1]) + 1/(N - n[2]))) * θ_2^2
# varV__u_12_coeff_hyp =  (N/((N-n[1])*(N-n[2])) + 3/N - 2*(1/(N-n[1]) + 1/(N-n[2]))) * (3*θ_2^2 - 2*θ_4) + (2 + 9/N - (2*n[1]*n[2]+(N+3)*(n[1]+n[2])-N*(2*N+3))/((N-n[1])*(N-n[2]))) * θ_2^2
# varV__u_12_coeff_hyp =  (N/((N-n[1])*(N-n[2])) + 3/N - 2*(1/(N-n[1]) + 1/(N-n[2]))) * (3*θ_2^2 - 2*θ_4) + (2 + 9/N - (2*n[1]*n[2]+(N+3)*(n[1]+n[2])-N*(2*N+3))/((N-n[1])*(N-n[2]))) * θ_2^2
# varV__u_12_coeff_hyp =  (N/((N-n[1])*(N-n[2])) + 3/N - 2*(1/(N-n[1]) + 1/(N-n[2]))) * (3*θ_2^2 - 2*θ_4) + 0 * θ_2^2
varV__u_12_coeff_hyp =  (N/((N-n[1])*(N-n[2])) + 3/N - 2*(1/(N-n[1]) + 1/(N-n[2]))) * (3*θ_2^2 - 2*θ_4) + ((4+3/N)*((n[1]*n[2])/((N-n[1])*(N-n[2]))) - 3/N*(n[1]/(N-n[1]) + n[2]/(N-n[2]))) * θ_2^2
zero = (varV__u_12_coeff - varV__u_12_coeff_hyp).full_simplify()
zero

Now that we have a few coefficients done, let's try to build the entire $\text{var}(V)$ hypothesis.

In [ ]:
def u_coeff(i, j):
    r"These coefficients assume we make the sum $i\le j$. Otherwise, you should divide the off-diagonal elements by 2."
    if i == j:
        return ((N/(n[i]*(N-n[i])) - 3/N) * θ_4 + ((2*n[i]-3)*(1/n[i] - 1/N) + 2*n[i]/N + 3*(1/N - n[i]/(N*(N-n[i])))) * θ_2^2)
    else:
        return ((N/((N-n[i])*(N-n[j])) + 3/N - 2*(1/(N-n[i]) + 1/(N-n[j]))) * (3*θ_2^2 - 2*θ_4) + ((4+3/N)*((n[i]*n[j])/((N-n[i])*(N-n[j]))) - 3/N*(n[i]/(N-n[i]) + n[j]/(N-n[j]))) * θ_2^2)
# Λ = sum([1 / n_ for n_ in n.values()]) - 3 / N
# varVhyp = (Λ * θ_4 + (2 - 3 * Λ) * θ_2^2) * (sum(u.values()))^2
varVhyp = sum([
    u_coeff(i, j) * u[i] * u[j]
    for i in range(1, G+1)
    for j in range(i, G+1)
])
varVhyp = varVhyp.full_simplify()
show(varVhyp)

### Test hypothesis

In [ ]:
zero = (varV - varVhyp).full_simplify()
zero

In [ ]:
varV1 = varV.factor()
show(varV1)

In [ ]:
varVnum = varV1 * N * product([n[i] * (N - n[i]) for i in range(1, G+1)])
varVnum = varVnum.full_simplify()
show(varVnum)

In [ ]:
nn = var('nn')

In [ ]:
varV1 = varV

varV1 = varV1.substitute(N == nn)
for g in range(1, G+1):
    varV1 = varV1.substitute(N - n[g] == nn - n[g])
varV1

In [ ]:
varV2 = varV1.factor()
varV2

In [ ]:
eV.dump('eV')
varV.dump('varV')
varV1.dump('varV1')

In [ ]:
for v in ['eV', 'varV', 'varV1']:
    globals()[v] = load(f'{v}.sobj')

In [ ]:
varV1

In [ ]:
varV2 = varV1

for g in range(1, G+1):
    for h in range(1, G+1):
        if g != h:
            varV2 = varV2.substitute(N - n[g] - n[h] == nn - n[g] - n[h])
varV2

## Calculate $\text{Var}(V_u)$

In [77]:
G = 4
NN, N, X, Xbar, u, V = get_NN_N_X_Xbar_u_V(G)
varV = expectation(V^2) - expectation(V)^2
varV = varV.simplify()
varV

4*(μ^3 + 3*μ*σ^2/N_1 + γ*σ^3/N_1^2)*(N_1*u_1*(N_1/(N_1 + N_2 + N_3 + N_4) - 1)^2/(N_2 + N_3 + N_4) + N_1^2*N_2*u_2/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_3 + N_4)) + N_1^2*N_3*u_3/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_4)) + N_1^2*N_4*u_4/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_3)))*(N_1*N_2*u_1*(N_1/(N_1 + N_2 + N_3 + N_4) - 1)/((N_1 + N_2 + N_3 + N_4)*(N_2 + N_3 + N_4)) + N_1*N_2*u_2*(N_2/(N_1 + N_2 + N_3 + N_4) - 1)/((N_1 + N_2 + N_3 + N_4)*(N_1 + N_3 + N_4)) + N_1*N_2*N_3*u_3/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_4)) + N_1*N_2*N_4*u_4/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_3)))*N^2*μ + 4*(μ^3 + 3*μ*σ^2/N_2 + γ*σ^3/N_2^2)*(N_2*u_2*(N_2/(N_1 + N_2 + N_3 + N_4) - 1)^2/(N_1 + N_3 + N_4) + N_1*N_2^2*u_1/((N_1 + N_2 + N_3 + N_4)^2*(N_2 + N_3 + N_4)) + N_2^2*N_3*u_3/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_4)) + N_2^2*N_4*u_4/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_3)))*(N_1*N_2*u_1*(N_1/(N_1 + N_2 + N_3 + N_4) - 1)/((N_1 + N_2 + N_3 + N_4)*(N_2 + N_3 + N_4)) + N_1*N_2*u_2*(N_

In [78]:
varV = varV.substitute(NN == sum(N.values())).simplify()
varV

4*(μ^3 + 3*μ*σ^2/N_1 + γ*σ^3/N_1^2)*(N_1*u_1*(N_1/(N_1 + N_2 + N_3 + N_4) - 1)^2/(N_2 + N_3 + N_4) + N_1^2*N_2*u_2/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_3 + N_4)) + N_1^2*N_3*u_3/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_4)) + N_1^2*N_4*u_4/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_3)))*(N_1*N_2*u_1*(N_1/(N_1 + N_2 + N_3 + N_4) - 1)/((N_1 + N_2 + N_3 + N_4)*(N_2 + N_3 + N_4)) + N_1*N_2*u_2*(N_2/(N_1 + N_2 + N_3 + N_4) - 1)/((N_1 + N_2 + N_3 + N_4)*(N_1 + N_3 + N_4)) + N_1*N_2*N_3*u_3/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_4)) + N_1*N_2*N_4*u_4/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_3)))*(N_1 + N_2 + N_3 + N_4)^2*μ + 4*(μ^3 + 3*μ*σ^2/N_2 + γ*σ^3/N_2^2)*(N_2*u_2*(N_2/(N_1 + N_2 + N_3 + N_4) - 1)^2/(N_1 + N_3 + N_4) + N_1*N_2^2*u_1/((N_1 + N_2 + N_3 + N_4)^2*(N_2 + N_3 + N_4)) + N_2^2*N_3*u_3/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_4)) + N_2^2*N_4*u_4/((N_1 + N_2 + N_3 + N_4)^2*(N_1 + N_2 + N_3)))*(N_1*N_2*u_1*(N_1/(N_1 + N_2 + N_3 + N_4) - 1)/((N_1 + N_2 + N_3 + N_4)*(N_2 + N_3 + N

In [79]:
varV = varV.full_simplify()
varV

(2*((N_1*N_2*N_3^2 + (N_1^2*N_2 + N_1*N_2^2)*N_3)*N_4^5 + 3*(N_1*N_2*N_3^3 + 2*(N_1^2*N_2 + N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3)*N_4^4 + (3*N_1*N_2*N_3^4 + 10*(N_1^2*N_2 + N_1*N_2^2)*N_3^3 + (10*N_1^3*N_2 + 21*N_1^2*N_2^2 + 10*N_1*N_2^3)*N_3^2 + (3*N_1^4*N_2 + 10*N_1^3*N_2^2 + 10*N_1^2*N_2^3 + 3*N_1*N_2^4)*N_3)*N_4^3 + (N_1*N_2*N_3^5 + 6*(N_1^2*N_2 + N_1*N_2^2)*N_3^4 + (10*N_1^3*N_2 + 21*N_1^2*N_2^2 + 10*N_1*N_2^3)*N_3^3 + 3*(2*N_1^4*N_2 + 7*N_1^3*N_2^2 + 7*N_1^2*N_2^3 + 2*N_1*N_2^4)*N_3^2 + (N_1^5*N_2 + 6*N_1^4*N_2^2 + 10*N_1^3*N_2^3 + 6*N_1^2*N_2^4 + N_1*N_2^5)*N_3)*N_4^2 + ((N_1^2*N_2 + N_1*N_2^2)*N_3^5 + 3*(N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3^4 + (3*N_1^4*N_2 + 10*N_1^3*N_2^2 + 10*N_1^2*N_2^3 + 3*N_1*N_2^4)*N_3^3 + (N_1^5*N_2 + 6*N_1^4*N_2^2 + 10*N_1^3*N_2^3 + 6*N_1^2*N_2^4 + N_1*N_2^5)*N_3^2 + (N_1^5*N_2^2 + 3*N_1^4*N_2^3 + 3*N_1^3*N_2^4 + N_1^2*N_2^5)*N_3)*N_4)*u_1^2 + 4*((N_1^2*N_2^2*N_3^2 + (N_1^3*N_2^2 + N_1^2*N_2^3)*N_3)*N_4^3 + (N_1^2*N_2^2*

# Optimization

In [103]:
λ = var('λ')

objective = varV - λ*(sum(u.values()) - 1)
# show(objective)

In [104]:
equations = []
variables = [*u.values(), λ]
for variable in variables:
    equations.append(diff(objective, variable) == 0)
# for equation in equations:
#     show(equation)
sol = solve(equations, variables)

In [91]:
u_1__1 = sol[0][0].rhs()
u_1__1

1/2*(8*N_1*N_2*N_3*N_4^5 + 8*(4*N_1*N_2*N_3^2 + (3*N_1^2*N_2 + 4*N_1*N_2^2)*N_3)*N_4^4 + 24*(2*N_1*N_2*N_3^3 + (3*N_1^2*N_2 + 4*N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 3*N_1^2*N_2^2 + 2*N_1*N_2^3)*N_3)*N_4^3 + (N_1^4*N_2 + N_1^3*N_2^2 - N_1^2*N_2^3 - N_1*N_2^4 - N_1^2*N_3^3 - N_1*N_3^4 - N_1^2*N_4^3 - N_1*N_4^4 + (N_1^3 + N_1^2*N_2 + 2*N_1*N_2^2)*N_3^2 + (N_1^3 + N_1^2*N_2 + 2*N_1*N_2^2 + N_1^2*N_3 + 2*N_1*N_3^2)*N_4^2 + (N_1^4 + 2*N_1^3*N_2 + N_1^2*N_2^2)*N_3 + (N_1^4 + 2*N_1^3*N_2 + N_1^2*N_2^2 + N_1^2*N_3^2 + 2*(N_1^3 + 3*N_1^2*N_2)*N_3)*N_4)*κ^3 + 8*(4*N_1*N_2*N_3^4 + 3*(3*N_1^2*N_2 + 4*N_1*N_2^2)*N_3^3 + 6*(N_1^3*N_2 + 3*N_1^2*N_2^2 + 2*N_1*N_2^3)*N_3^2 + (N_1^4*N_2 + 6*N_1^3*N_2^2 + 9*N_1^2*N_2^3 + 4*N_1*N_2^4)*N_3)*N_4^2 + 2*(N_1^4*N_2^2 + 3*N_1^3*N_2^3 + 3*N_1^2*N_2^4 + N_1*N_2^5 + N_1*N_3^5 + N_1*N_4^5 + (3*N_1^2 + N_1*N_2)*N_3^4 + (3*N_1^2 + N_1*N_2 + N_1*N_3)*N_4^4 + (3*N_1^3 + 4*N_1^2*N_2 - 2*N_1*N_2^2)*N_3^3 + (3*N_1^3 + 4*N_1^2*N_2 - 2*N_1*N_2^2 - 2*N_1*N_3^2 + 4*(N_1^2 + N_1*N_2

In [92]:
u_1__1 = u_1__1.full_simplify()
u_1__1

1/2*(8*N_1*N_2*N_3*N_4^5 + 8*(4*N_1*N_2*N_3^2 + (3*N_1^2*N_2 + 4*N_1*N_2^2)*N_3)*N_4^4 + 24*(2*N_1*N_2*N_3^3 + (3*N_1^2*N_2 + 4*N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 3*N_1^2*N_2^2 + 2*N_1*N_2^3)*N_3)*N_4^3 + (N_1^4*N_2 + N_1^3*N_2^2 - N_1^2*N_2^3 - N_1*N_2^4 - N_1^2*N_3^3 - N_1*N_3^4 - N_1^2*N_4^3 - N_1*N_4^4 + (N_1^3 + N_1^2*N_2 + 2*N_1*N_2^2)*N_3^2 + (N_1^3 + N_1^2*N_2 + 2*N_1*N_2^2 + N_1^2*N_3 + 2*N_1*N_3^2)*N_4^2 + (N_1^4 + 2*N_1^3*N_2 + N_1^2*N_2^2)*N_3 + (N_1^4 + 2*N_1^3*N_2 + N_1^2*N_2^2 + N_1^2*N_3^2 + 2*(N_1^3 + 3*N_1^2*N_2)*N_3)*N_4)*κ^3 + 8*(4*N_1*N_2*N_3^4 + 3*(3*N_1^2*N_2 + 4*N_1*N_2^2)*N_3^3 + 6*(N_1^3*N_2 + 3*N_1^2*N_2^2 + 2*N_1*N_2^3)*N_3^2 + (N_1^4*N_2 + 6*N_1^3*N_2^2 + 9*N_1^2*N_2^3 + 4*N_1*N_2^4)*N_3)*N_4^2 + 2*(N_1^4*N_2^2 + 3*N_1^3*N_2^3 + 3*N_1^2*N_2^4 + N_1*N_2^5 + N_1*N_3^5 + N_1*N_4^5 + (3*N_1^2 + N_1*N_2)*N_3^4 + (3*N_1^2 + N_1*N_2 + N_1*N_3)*N_4^4 + (3*N_1^3 + 4*N_1^2*N_2 - 2*N_1*N_2^2)*N_3^3 + (3*N_1^3 + 4*N_1^2*N_2 - 2*N_1*N_2^2 - 2*N_1*N_3^2 + 4*(N_1^2 + N_1*N_2

In [93]:
u_1__2 = u_1__1.factor()
show(u_1__2)

1/2*(2*N_1*N_2 + 2*N_2^2 + 2*N_2*N_3 + 2*N_2*N_4 + N_1*κ - N_2*κ + N_3*κ + N_4*κ)*(2*N_1*N_3 + 2*N_2*N_3 + 2*N_3^2 + 2*N_3*N_4 + N_1*κ + N_2*κ - N_3*κ + N_4*κ)*(2*N_1*N_4 + 2*N_2*N_4 + 2*N_3*N_4 + 2*N_4^2 + N_1*κ + N_2*κ + N_3*κ - N_4*κ)*N_1*(N_2 + N_3 + N_4)/((12*N_1^4*N_2*N_3*N_4 + 36*N_1^3*N_2^2*N_3*N_4 + 36*N_1^2*N_2^3*N_3*N_4 + 12*N_1*N_2^4*N_3*N_4 + 36*N_1^3*N_2*N_3^2*N_4 + 72*N_1^2*N_2^2*N_3^2*N_4 + 36*N_1*N_2^3*N_3^2*N_4 + 36*N_1^2*N_2*N_3^3*N_4 + 36*N_1*N_2^2*N_3^3*N_4 + 12*N_1*N_2*N_3^4*N_4 + 36*N_1^3*N_2*N_3*N_4^2 + 72*N_1^2*N_2^2*N_3*N_4^2 + 36*N_1*N_2^3*N_3*N_4^2 + 72*N_1^2*N_2*N_3^2*N_4^2 + 72*N_1*N_2^2*N_3^2*N_4^2 + 36*N_1*N_2*N_3^3*N_4^2 + 36*N_1^2*N_2*N_3*N_4^3 + 36*N_1*N_2^2*N_3*N_4^3 + 36*N_1*N_2*N_3^2*N_4^3 + 12*N_1*N_2*N_3*N_4^4 + 4*N_1^4*N_2*N_3*κ + 12*N_1^3*N_2^2*N_3*κ + 12*N_1^2*N_2^3*N_3*κ + 4*N_1*N_2^4*N_3*κ + 12*N_1^3*N_2*N_3^2*κ + 24*N_1^2*N_2^2*N_3^2*κ + 12*N_1*N_2^3*N_3^2*κ + 12*N_1^2*N_2*N_3^3*κ + 12*N_1*N_2^2*N_3^3*κ + 4*N_1*N_2*N_3^4*κ + 4*N_1^4*N_2*N_4*κ + 12*N_1^3*N_2^2*N_4*κ + 12*N_1^2*N_2^3*N_4*κ + 4*N_1*N_2^4*N_4*κ + 4*N_1^4*N_3*N_4*κ + 12*N_1^3*N_2*N_3*N_4*κ + 16*N_1^2*N_2^2*N_3*N_4*κ + 12*N_1*N_2^3*N_3*N_4*κ + 4*N_2^4*N_3*N_4*κ + 12*N_1^3*N_3^2*N_4*κ + 16*N_1^2*N_2*N_3^2*N_4*κ + 16*N_1*N_2^2*N_3^2*N_4*κ + 12*N_2^3*N_3^2*N_4*κ + 12*N_1^2*N_3^3*N_4*κ + 12*N_1*N_2*N_3^3*N_4*κ + 12*N_2^2*N_3^3*N_4*κ + 4*N_1*N_3^4*N_4*κ + 4*N_2*N_3^4*N_4*κ + 12*N_1^3*N_2*N_4^2*κ + 24*N_1^2*N_2^2*N_4^2*κ + 12*N_1*N_2^3*N_4^2*κ + 12*N_1^3*N_3*N_4^2*κ + 16*N_1^2*N_2*N_3*N_4^2*κ + 16*N_1*N_2^2*N_3*N_4^2*κ + 12*N_2^3*N_3*N_4^2*κ + 24*N_1^2*N_3^2*N_4^2*κ + 16*N_1*N_2*N_3^2*N_4^2*κ + 24*N_2^2*N_3^2*N_4^2*κ + 12*N_1*N_3^3*N_4^2*κ + 12*N_2*N_3^3*N_4^2*κ + 12*N_1^2*N_2*N_4^3*κ + 12*N_1*N_2^2*N_4^3*κ + 12*N_1^2*N_3*N_4^3*κ + 12*N_1*N_2*N_3*N_4^3*κ + 12*N_2^2*N_3*N_4^3*κ + 12*N_1*N_3^2*N_4^3*κ + 12*N_2*N_3^2*N_4^3*κ + 4*N_1*N_2*N_4^4*κ + 4*N_1*N_3*N_4^4*κ + 4*N_2*N_3*N_4^4*κ + N_1^4*N_2*κ^2 + 3*N_1^3*N_2^2*κ^2 + 3*N_1^2*N_2^3*κ^2 + N_1*N_2^4*κ^2 + N_1^4*N_3*κ^2 + 2*N_1^3*N_2*N_3*κ^2 + 2*N_1^2*N_2^2*N_3*κ^2 + 2*N_1*N_2^3*N_3*κ^2 + N_2^4*N_3*κ^2 + 3*N_1^3*N_3^2*κ^2 + 2*N_1^2*N_2*N_3^2*κ^2 + 2*N_1*N_2^2*N_3^2*κ^2 + 3*N_2^3*N_3^2*κ^2 + 3*N_1^2*N_3^3*κ^2 + 2*N_1*N_2*N_3^3*κ^2 + 3*N_2^2*N_3^3*κ^2 + N_1*N_3^4*κ^2 + N_2*N_3^4*κ^2 + N_1^4*N_4*κ^2 + 2*N_1^3*N_2*N_4*κ^2 + 2*N_1^2*N_2^2*N_4*κ^2 + 2*N_1*N_2^3*N_4*κ^2 + N_2^4*N_4*κ^2 + 2*N_1^3*N_3*N_4*κ^2 + 12*N_1^2*N_2*N_3*N_4*κ^2 + 12*N_1*N_2^2*N_3*N_4*κ^2 + 2*N_2^3*N_3*N_4*κ^2 + 2*N_1^2*N_3^2*N_4*κ^2 + 12*N_1*N_2*N_3^2*N_4*κ^2 + 2*N_2^2*N_3^2*N_4*κ^2 + 2*N_1*N_3^3*N_4*κ^2 + 2*N_2*N_3^3*N_4*κ^2 + N_3^4*N_4*κ^2 + 3*N_1^3*N_4^2*κ^2 + 2*N_1^2*N_2*N_4^2*κ^2 + 2*N_1*N_2^2*N_4^2*κ^2 + 3*N_2^3*N_4^2*κ^2 + 2*N_1^2*N_3*N_4^2*κ^2 + 12*N_1*N_2*N_3*N_4^2*κ^2 + 2*N_2^2*N_3*N_4^2*κ^2 + 2*N_1*N_3^2*N_4^2*κ^2 + 2*N_2*N_3^2*N_4^2*κ^2 + 3*N_3^3*N_4^2*κ^2 + 3*N_1^2*N_4^3*κ^2 + 2*N_1*N_2*N_4^3*κ^2 + 3*N_2^2*N_4^3*κ^2 + 2*N_1*N_3*N_4^3*κ^2 + 2*N_2*N_3*N_4^3*κ^2 + 3*N_3^2*N_4^3*κ^2 + N_1*N_4^4*κ^2 + N_2*N_4^4*κ^2 + N_3*N_4^4*κ^2 + N_1^2*N_2*N_3*κ^3 + N_1*N_2^2*N_3*κ^3 + N_1*N_2*N_3^2*κ^3 + N_1^2*N_2*N_4*κ^3 + N_1*N_2^2*N_4*κ^3 + N_1^2*N_3*N_4*κ^3 + N_2^2*N_3*N_4*κ^3 + N_1*N_3^2*N_4*κ^3 + N_2*N_3^2*N_4*κ^3 + N_1*N_2*N_4^2*κ^3 + N_1*N_3*N_4^2*κ^3 + N_2*N_3*N_4^2*κ^3)*(N_1 + N_2 + N_3 + N_4))

In [94]:
u_1_numer = u_1__2.numerator()
show(u_1_numer)

(2*N_1*N_2 + 2*N_2^2 + 2*N_2*N_3 + 2*N_2*N_4 + N_1*κ - N_2*κ + N_3*κ + N_4*κ)*(2*N_1*N_3 + 2*N_2*N_3 + 2*N_3^2 + 2*N_3*N_4 + N_1*κ + N_2*κ - N_3*κ + N_4*κ)*(2*N_1*N_4 + 2*N_2*N_4 + 2*N_3*N_4 + 2*N_4^2 + N_1*κ + N_2*κ + N_3*κ - N_4*κ)*N_1*(N_2 + N_3 + N_4)

In [95]:
Uhyp = dict()
allN = sum(N.values())
for g in range(1, G+1):
    Uhyp[g] = σ^4 * N[g] * (allN - N[g]) * product([
        (κ * (allN - 2*N[h]) + 2*allN*N[h])
        for h in range(1, G+1) if h != g
    ])
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
uhyp[1]

1/2*(2*(N_1 + N_2 + N_3 + N_4)*N_2 + (N_1 - N_2 + N_3 + N_4)*κ)*(2*(N_1 + N_2 + N_3 + N_4)*N_3 + (N_1 + N_2 - N_3 + N_4)*κ)*(2*(N_1 + N_2 + N_3 + N_4)*N_4 + (N_1 + N_2 + N_3 - N_4)*κ)*N_1*(N_2 + N_3 + N_4)/(12*N_1*N_2*N_3*N_4^5 + 48*(N_1*N_2*N_3^2 + (N_1^2*N_2 + N_1*N_2^2)*N_3)*N_4^4 + 72*(N_1*N_2*N_3^3 + 2*(N_1^2*N_2 + N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3)*N_4^3 + (N_1*N_2*N_3^3 + (N_1*N_2 + (N_1 + N_2)*N_3)*N_4^3 + 2*(N_1^2*N_2 + N_1*N_2^2)*N_3^2 + (2*N_1^2*N_2 + 2*N_1*N_2^2 + 2*(N_1 + N_2)*N_3^2 + (2*N_1^2 + 3*N_1*N_2 + 2*N_2^2)*N_3)*N_4^2 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3 + (N_1 + N_2)*N_3^3 + (2*N_1^2 + 3*N_1*N_2 + 2*N_2^2)*N_3^2 + (N_1^3 + 3*N_1^2*N_2 + 3*N_1*N_2^2 + N_2^3)*N_3)*N_4)*κ^3 + 48*(N_1*N_2*N_3^4 + 3*(N_1^2*N_2 + N_1*N_2^2)*N_3^3 + 3*(N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3^2 + (N_1^4*N_2 + 3*N_1^3*N_2^2 + 3*N_1^2*N_2^3 + N_1*N_2^4)*N_3)*N_4^2 + (N_1^5*N_2 + 4*N_1^4*N_2^2 + 6*N_1^3*N_2^3 

In [97]:
# Equivalent hypothesis
Uhyp = dict()
allN = sum(N.values())
for g in range(1, G+1):
    Uhyp[g] =(allN - N[g]) / (1 + κ * (1/(2*N[g]) - 1/allN))
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
show(Uhyp[1])
uhyp[1]

-2*(N_2 + N_3 + N_4)/(κ*(2/(N_1 + N_2 + N_3 + N_4) - 1/N_1) - 2)

-1/2*(16*N_1*N_2*N_3*N_4^5 + 64*(N_1*N_2*N_3^2 + (N_1^2*N_2 + N_1*N_2^2)*N_3)*N_4^4 - (N_1^4 - 2*N_1^2*N_2^2 + N_2^4 + N_3^4 - 8*N_1*N_2*N_3*N_4 + N_4^4 - 2*(N_1^2 + N_2^2)*N_3^2 - 2*(N_1^2 + N_2^2 + N_3^2)*N_4^2)*κ^4 + 96*(N_1*N_2*N_3^3 + 2*(N_1^2*N_2 + N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3)*N_4^3 + 2*(N_1^5 + N_1^4*N_2 - 2*N_1^3*N_2^2 - 2*N_1^2*N_2^3 + N_1*N_2^4 + N_2^5 + (N_1 + N_2)*N_3^4 + N_3^5 + (N_1 + N_2 + N_3)*N_4^4 + N_4^5 - 2*(N_1^2 - 2*N_1*N_2 + N_2^2)*N_3^3 - 2*(N_1^2 - 2*N_1*N_2 + N_2^2 - 2*(N_1 + N_2)*N_3 + N_3^2)*N_4^3 - 2*(N_1^3 - 3*N_1^2*N_2 - 3*N_1*N_2^2 + N_2^3)*N_3^2 - 2*(N_1^3 - 3*N_1^2*N_2 - 3*N_1*N_2^2 + N_2^3 - 3*(N_1 + N_2)*N_3^2 + N_3^3 - (3*N_1^2 + 2*N_1*N_2 + 3*N_2^2)*N_3)*N_4^2 + (N_1^4 + 4*N_1^3*N_2 + 6*N_1^2*N_2^2 + 4*N_1*N_2^3 + N_2^4)*N_3 + (N_1^4 + 4*N_1^3*N_2 + 6*N_1^2*N_2^2 + 4*N_1*N_2^3 + N_2^4 + 4*(N_1 + N_2)*N_3^3 + N_3^4 + 2*(3*N_1^2 + 2*N_1*N_2 + 3*N_2^2)*N_3^2 + 4*(N_1^3 + N_1^2*N_2 + N_1*N_2^2 + N_2^3)*N_3)*N_4)*κ^3 +

In [99]:
# Equivalent hypothesis 2
Uhyp = dict()
allN = sum(N.values())
for g in range(1, G+1):
    Uhyp[g] =(allN - N[g]) / (1 + 1/(2*N[g]) / (1/κ - 1/allN))
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
uhyp[1]

-1/2*(16*N_1*N_2*N_3*N_4^5 + 64*(N_1*N_2*N_3^2 + (N_1^2*N_2 + N_1*N_2^2)*N_3)*N_4^4 - (N_1^4 - 2*N_1^2*N_2^2 + N_2^4 + N_3^4 - 8*N_1*N_2*N_3*N_4 + N_4^4 - 2*(N_1^2 + N_2^2)*N_3^2 - 2*(N_1^2 + N_2^2 + N_3^2)*N_4^2)*κ^4 + 96*(N_1*N_2*N_3^3 + 2*(N_1^2*N_2 + N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3)*N_4^3 + 2*(N_1^5 + N_1^4*N_2 - 2*N_1^3*N_2^2 - 2*N_1^2*N_2^3 + N_1*N_2^4 + N_2^5 + (N_1 + N_2)*N_3^4 + N_3^5 + (N_1 + N_2 + N_3)*N_4^4 + N_4^5 - 2*(N_1^2 - 2*N_1*N_2 + N_2^2)*N_3^3 - 2*(N_1^2 - 2*N_1*N_2 + N_2^2 - 2*(N_1 + N_2)*N_3 + N_3^2)*N_4^3 - 2*(N_1^3 - 3*N_1^2*N_2 - 3*N_1*N_2^2 + N_2^3)*N_3^2 - 2*(N_1^3 - 3*N_1^2*N_2 - 3*N_1*N_2^2 + N_2^3 - 3*(N_1 + N_2)*N_3^2 + N_3^3 - (3*N_1^2 + 2*N_1*N_2 + 3*N_2^2)*N_3)*N_4^2 + (N_1^4 + 4*N_1^3*N_2 + 6*N_1^2*N_2^2 + 4*N_1*N_2^3 + N_2^4)*N_3 + (N_1^4 + 4*N_1^3*N_2 + 6*N_1^2*N_2^2 + 4*N_1*N_2^3 + N_2^4 + 4*(N_1 + N_2)*N_3^3 + N_3^4 + 2*(3*N_1^2 + 2*N_1*N_2 + 3*N_2^2)*N_3^2 + 4*(N_1^3 + N_1^2*N_2 + N_1*N_2^2 + N_2^3)*N_3)*N_4)*κ^3 +

In [101]:
# NOT!!!!!! Equivalent hypothesis 3 (NOT EQUIVALENT!)
Uhyp = dict()
allN = sum(N.values())
for g in range(1, G+1):
    Uhyp[g] =(allN - N[g]) * (1 + 1/(1 - N[g]*(1/κ - 1/allN)))
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
uhyp[1]

(N_1*N_2*N_3*N_4^5 + 4*(N_1*N_2*N_3^2 + (N_1^2*N_2 + N_1*N_2^2)*N_3)*N_4^4 + (2*N_1^4 + 9*N_1^3*N_2 + 14*N_1^2*N_2^2 + 9*N_1*N_2^3 + 2*N_2^4 + 9*(N_1 + N_2)*N_3^3 + 2*N_3^4 + 9*(N_1 + N_2 + N_3)*N_4^3 + 2*N_4^4 + 2*(7*N_1^2 + 15*N_1*N_2 + 7*N_2^2)*N_3^2 + 2*(7*N_1^2 + 15*N_1*N_2 + 7*N_2^2 + 15*(N_1 + N_2)*N_3 + 7*N_3^2)*N_4^2 + 3*(3*N_1^3 + 10*N_1^2*N_2 + 10*N_1*N_2^2 + 3*N_2^3)*N_3 + (9*N_1^3 + 30*N_1^2*N_2 + 30*N_1*N_2^2 + 9*N_2^3 + 30*(N_1 + N_2)*N_3^2 + 9*N_3^3 + 5*(6*N_1^2 + 13*N_1*N_2 + 6*N_2^2)*N_3)*N_4)*κ^4 + 6*(N_1*N_2*N_3^3 + 2*(N_1^2*N_2 + N_1*N_2^2)*N_3^2 + (N_1^3*N_2 + 2*N_1^2*N_2^2 + N_1*N_2^3)*N_3)*N_4^3 - (N_1^5 + 7*N_1^4*N_2 + 16*N_1^3*N_2^2 + 16*N_1^2*N_2^3 + 7*N_1*N_2^4 + N_2^5 + 7*(N_1 + N_2)*N_3^4 + N_3^5 + 7*(N_1 + N_2 + N_3)*N_4^4 + N_4^5 + (16*N_1^2 + 37*N_1*N_2 + 16*N_2^2)*N_3^3 + (16*N_1^2 + 37*N_1*N_2 + 16*N_2^2 + 37*(N_1 + N_2)*N_3 + 16*N_3^2)*N_4^3 + 4*(4*N_1^3 + 15*N_1^2*N_2 + 15*N_1*N_2^2 + 4*N_2^3)*N_3^2 + (16*N_1^3 + 60*N_1^2*N_2 + 60*N_1*N_2^2 + 16*N_2

### Test hypothesis for $u_1$

In [102]:
zero = (u_1__1 - uhyp[1]).full_simplify()
zero

1/2*((12*N_1^8*N_2 + 63*N_1^7*N_2^2 + 117*N_1^6*N_2^3 + 66*N_1^5*N_2^4 - 66*N_1^4*N_2^5 - 117*N_1^3*N_2^6 - 63*N_1^2*N_2^7 - 12*N_1*N_2^8 - 12*N_1*N_3^8 - 12*N_1*N_4^8 - (63*N_1^2 + 59*N_1*N_2)*N_3^7 - (63*N_1^2 + 59*N_1*N_2 + (59*N_1 + 8*N_2)*N_3)*N_4^7 - (117*N_1^3 + 239*N_1^2*N_2 + 106*N_1*N_2^2)*N_3^6 - (117*N_1^3 + 239*N_1^2*N_2 + 106*N_1*N_2^2 + 2*(53*N_1 + 26*N_2)*N_3^2 + (239*N_1^2 + 280*N_1*N_2 + 52*N_2^2)*N_3)*N_4^6 - (66*N_1^4 + 272*N_1^3*N_2 + 287*N_1^2*N_2^2 + 85*N_1*N_2^3)*N_3^5 - (66*N_1^4 + 272*N_1^3*N_2 + 287*N_1^2*N_2^2 + 85*N_1*N_2^3 + 17*(5*N_1 + 8*N_2)*N_3^3 + (287*N_1^2 + 564*N_1*N_2 + 280*N_2^2)*N_3^2 + (272*N_1^3 + 731*N_1^2*N_2 + 564*N_1*N_2^2 + 136*N_2^3)*N_3)*N_4^5 + (66*N_1^5 + 118*N_1^4*N_2 - 7*N_1^3*N_2^2 - 107*N_1^2*N_2^3 - 52*N_1*N_2^4)*N_3^4 + (66*N_1^5 + 118*N_1^4*N_2 - 7*N_1^3*N_2^2 - 107*N_1^2*N_2^3 - 52*N_1*N_2^4 - 4*(13*N_1 + 46*N_2)*N_3^4 - (107*N_1^2 + 705*N_1*N_2 + 580*N_2^2)*N_3^3 - (7*N_1^3 + 733*N_1^2*N_2 + 1322*N_1*N_2^2 + 580*N_2^3)*N_3^2 +

## Construct varV for various $u_i$

In [ ]:
# Equivalent hypothesis
Uhyp = dict()
for g in range(1, G+1):
    Uhyp[g] =(N - n[g]) / (1 + κ * (1/(2*n[g]) - 1/N))
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
Voptimal_num = N * sum([Uhyp[g] * n[g] / (N - n[g]) * S[g]^2 for g in range(1, G+1)])
Voptimal = Voptimal_num / Uhyp_sum
show(Voptimal_num)
show(Uhyp_sum)

In [ ]:
Uhyp = dict()
for g in range(1, G+1):
    Uhyp[g] =(N - n[g]) / (1 + 1/(2*n[g]) / (1/κ - 1/N))
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
Voptimal = N * sum([Uhyp[g] * n[g] / (N - n[g]) * S[g]^2 for g in range(1, G+1)]) / Uhyp_sum
Voptimal

In [ ]:
# κ = 0 solution
Uhyp0 = dict()
for g in range(1, G+1):
       Uhyp0[g] = (N - n[g])
Uhyp0_sum = sum(Uhyp0.values()).full_simplify()
uhyp0 = {g: Uhyp0_g / Uhyp0_sum for g, Uhyp0_g in Uhyp0.items()}
V0optimal = N * sum([Uhyp0[g] * n[g] / (N - n[g]) * S[g] for g in range(1, G+1)]) / Uhyp0_sum
show(V0optimal)
show(uhyp0[1])

In [ ]:
# 1st order solution in  κN/(2N_g(κ-N))
Uhyp = dict()
for g in range(1, G+1):
    Uhyp[g] =(N - n[g]) * (1 + 1/(2*n[g]) / (1/N - 1/κ))
Uhyp_sum = sum(Uhyp.values()).full_simplify()
uhyp = {g: Uhyp_g / Uhyp_sum for g, Uhyp_g in Uhyp.items()}
# Squared differences S[g] = (Xbar[g]-Xbarbar)^2
S = get_regular_variables(G, prefix='S_')
V_1o_num = N * sum([Uhyp[g] * n[g] / (N - n[g]) * S[g] for g in range(1, G+1)])
V_1o = V_1o_num / Uhyp_sum
show(V_1o_num)
show(Uhyp_sum)

In [ ]:
V_1o = V_1o.full_simplify()
V_1o

In [ ]:
V_1o_f = V_1o.factor()
V_1o_n = V_1o_f.numerator()
V_1o_d = V_1o_f.denominator()
show(V_1o_n)
show(V_1o_d)

In [ ]:
show(V_1o_n.coefficient(S[1]))

In [ ]:
V_1o_d = V_1o_d.full_simplify()
show(V_1o_d)

In [ ]:
V_1o_d1 = V_1o_d.substitute(n[4] == nn - (N-n[4]).full_simplify()).full_simplify()
show(V_1o_d1)

In [ ]:
sum_n_inv = var('sum_n_inv')
V_1o_hyp = N * sum([(2*n[g] - κ *(1 + 2*n[g]/nn)) * S[g] for g in range(1, G+1)]) / (6 - κ*(2/nn + sum_n_inv))
V_1o

## scratch

In [ ]:
x = var('x')
(1-3*x+3*x^2).factor()

## Augmented calculations

In [ ]:
N, n_g, n_h = var("N n_g n_h")
parent_ring = (n_g/n_h).parent()
P.<X_g, X_h, X_gh> = parent_ring[]
X_g^3*X_h

In [ ]:
Xbar = n_g * X_g + n_h * X_h + (1 - n_g - n_h) * X_gh
cross_term = (X_g - Xbar)^2 * (X_h - Xbar)^2
cross_term

In [ ]:
def expectation(expression):
    "expression should be an element of P"
    sizes = {
        0: N*n_g, # X_g
        1: N*n_h, # X_h
        2: N*(1-n_g-n_h) # X_gh
    }
    result = 0
    for coefficient, monomial in expression:
        new_term = coefficient
        for index, exponent in enumerate(monomial.exponents()[0]):
            new_term *= sample_moment(exponent, sizes[index])
        result += new_term
    return result
show(expectation(X_g*X_h*X_gh^2))

In [ ]:
cross_term_exp = expectation(cross_term)
cross_term_exp = cross_term_exp.full_simplify()
cross_term_exp

In [ ]:
fg = 1/(1-n_g)
fh = 1/(1-n_h)
zero = (
    fg/n_g*((1-3*n_g+3*n_g^2)*κ + 3*N*n_g*(1-n_g))
) - (
    3*N + (fg/n_g - 3)*κ
)
zero = zero.full_simplify()
show(zero)

In [ ]:
fg = 1/(1-n_g)
fh = 1/(1-n_h)
zero = (
    fg*fh*((n_g+n_h-3*n_g*n_h)*κ + N*(1-n_g-n_h+3*n_g*n_h))
) - (
    N*(1+2*(fg-1)*(fh-1)) + (1-(fg-2)*(fh-2))*κ
)
zero = zero.full_simplify()
show(zero)

In [ ]:
dir(terms[0])

In [ ]:
terms[-1]

In [ ]:
terms[-1].exponents()

In [ ]:
from sage.symbolic.expression_conversions import RingConverter
E = RingConverter(P, subs_dict={
    X_g^4: var('K')
})

In [ ]:
E(X_g^4)

## Auxiliary scratch

In [ ]:
N = var('N')
n(g) = 1/N

In [ ]:
Λ = var('Λ')
q = (sum((2*Λ*(1-n(g)-2*(κ-N))/((1-2*n(g))*κ + 2*n(g)*N)*n(g)/(1-2*n(g))), g, 1, N))/(1-(κ-2*N)*sum(1/((1-2*n(g))*κ + 2*n(g)*N)*n(g)/(1-2*n(g)), g, 1, N))
show(q)

In [ ]:
q = q.full_simplify()

In [ ]:
show(q)

In [ ]:
show(q.factor())

In [ ]:
u(g) = n(g)*(1-n(g))/(1-2*n(g))*(2*Λ*(1-n(g)) + ((q-2)*κ-2*(q-1)*N))/((1-2*n(g))*κ+2*n(g)*N)

In [ ]:
u(g).full_simplify().collect(N).factor().show()

In [ ]:
u(g) = 1/N

In [ ]:
varV = σ^4/N*(
    (1-(sum(u(g)/(1-n(g)), g, 1, N) - 2)^2)*κ + 2*(sum(u(g)/(1-n(g)), g, 1, N) - 1)^2*N
    + sum(u(g)^2*(
        (1/n(g)/(1-n(g))-3-1+(1/(1-n(g)) - 2)^2) * κ
        + (3-1-2*(1/(1-n(g)) - 1)^2) * N
    ), g, 1, N)
)
varV = varV.full_simplify().factor()
show(varV)